## Tasks
```
1. Data Exploration and Preprocessing
●	Begin by loading and exploring the "Alphabets_data.csv" dataset. Summarize its key features such as the number of samples, features, and
classes.

●	Execute necessary data preprocessing steps including data normalization, managing missing values.

2. Model Implementation
●	Construct a basic ANN model using your chosen high-level neural network library. Ensure your model includes at least one hidden layer.
●	Divide the dataset into training and test sets.
●	Train your model on the training set and then use it to make predictions on the test set.

3. Hyperparameter Tuning
●	Modify various hyperparameters, such as the number of hidden layers, neurons per hidden layer, activation functions, and learning rate, to observe their impact on model performance.
●	Adopt a structured approach like grid search or random search for hyperparameter tuning, documenting your methodology thoroughly.

4. Evaluation
●	Employ suitable metrics such as accuracy, precision, recall, and F1-score to evaluate your model's performance.
●	Discuss the performance differences between the model with default hyperparameters and the tuned model, emphasizing the effects of hyperparameter tuning.

Evaluation Criteria
●	Accuracy and completeness of the implementation.
●	Proficiency in data preprocessing and model development.
●	Systematic approach and thoroughness in hyperparameter tuning.
●	Depth of evaluation and discussion.
●	Overall quality of the report.
```

## Answers

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Loading data to our enviroment
dataset = pd.read_csv('sonardataset.csv')

# Working on a copy
df = dataset.copy()

#Understanding Data and extraction information from it 
print("\n<-----------INFO---------->\n")
print(df.info())

print("\n<------------DESCRIBE----------->\n")
print(df.describe())

print("\n<------------DESCRIBE ALL CATEGORICAL AND NUMERICAL VALUES----------->\n")
print(df.describe(include='all'))

print("\n<------------CHCKING NULL VALUES----------->\n")
print(df.isnull().sum().to_string())

## Data Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder


# Dividing data into target and independent varibales
X= df.iloc[:,:-1]
y =df.iloc[:,-1]

# label encoding for the output categorical data
l_en = LabelEncoder()
y_enc = l_en.fit_transform(y)

# Standardizing the numerical variable data (scaling features)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("shape of the scaled data", X_scaled.shape)

## Model Building And HyperparameterTunning

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
import tensorflow as tf
import random

seed = 42
np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)

#Splitting the data into train and test 
X_train, X_test, y_train, y_test = train_test_split(X_scaled,y_enc,test_size=0.20,random_state=seed,stratify=y_enc)



In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV

def build_model(n_hidden=1, n_neurons=32, activation='relu', learning_rate=0.001):
    model = Sequential()
    # First layer with input shape
    model.add(Dense(n_neurons, activation=activation, input_shape=(X_train.shape[1],)))
    # Hidden layers
    for _ in range(n_hidden - 1):
        model.add(Dense(n_neurons, activation=activation))
    # Output layer
    model.add(Dense(1, activation='sigmoid'))

    opt = Adam(learning_rate=learning_rate)
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    return model

keras_clf = KerasClassifier(build_fn=build_model, verbose=0)

parameters = {
    "epochs": [50, 100],
    "batch_size": [8, 16],
    "validation_split": [0.2],
}


neural_model = GridSearchCV(estimator=keras_clf,param_grid=parameters,scoring="accuracy",cv=3,verbose=2)
neural_networks_model = neural_model.fit(X_train, y_train)

In [ ]:
# Best parameters
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report

print("Best parameters found: ", neural_networks_model.best_params_)
print("Best cross-val accuracy: ", neural_networks_model.best_score_)

# Evaluate on test set
best_model = neural_networks_model.best_estimator_

y_pred = (best_model.predict(X_test))

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Test Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1-score:", f1)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
